# BenignIDS — 07-CNN

Benchmark a simple CNN on payload sequence features. Conforms to Style Guide: 7.1 loader, 7.2 training, 7.3 comparison.

## Section 7.1 — CNN Input Loader

**What this does**
- Loads sparse payload features (`payload_seq_train.npz`, `payload_seq_val.npz`).
- Hydrates labels from split artefacts.
- Sets `CNN_AVAILABLE` flag.

**Why**
- Keeps CNN benchmark optional but reproducible.

In [1]:
# ======================================================
# Section 7.1 — CNN Input Loader
# ======================================================
print(">>> Section 7.1 — CNN Input Loader: start")

from pathlib import Path
import numpy as np
import warnings

def _maybe_import_pandas():
    try:
        import pandas as pd
        return pd
    except Exception:
        return None

def _hydrate_labels(split_dir: Path, target_col: str = "label"):
    pd = _maybe_import_pandas()
    y_train = y_val = None
    candidates = {
        "train": [split_dir/"y_train.parquet", split_dir/"y_train.csv",
                  split_dir/"y_train.npy", split_dir/"y_train.pkl"],
        "val":   [split_dir/"y_val.parquet",   split_dir/"y_val.csv",
                  split_dir/"y_val.npy",   split_dir/"y_val.pkl"],
    }
    for c in candidates["train"]:
        try:
            if c.suffix == ".parquet" and c.exists() and pd is not None:
                df = pd.read_parquet(c); y_train = df[target_col].values; break
            if c.suffix == ".csv" and c.exists() and pd is not None:
                df = pd.read_csv(c); y_train = df[target_col].values; break
            if c.suffix == ".npy" and c.exists():
                y_train = np.load(c); break
            if c.suffix == ".pkl" and c.exists() and pd is not None:
                obj = pd.read_pickle(c); y_train = getattr(obj, "values", obj); break
        except Exception: pass
    for c in candidates["val"]:
        try:
            if c.suffix == ".parquet" and c.exists() and pd is not None:
                df = pd.read_parquet(c); y_val = df[target_col].values; break
            if c.suffix == ".csv" and c.exists() and pd is not None:
                df = pd.read_csv(c); y_val = df[target_col].values; break
            if c.suffix == ".npy" and c.exists():
                y_val = np.load(c); break
            if c.suffix == ".pkl" and c.exists() and pd is not None:
                obj = pd.read_pickle(c); y_val = getattr(obj, "values", obj); break
        except Exception: pass
    return (y_train, y_val)

def _load_seq():
    import scipy.sparse as sp
    stage_root = Path(globals().get("STAGE_ROOT", "staging"))
    seq_dir = stage_root / "payload_seq_preproc"
    seq_Xtr, seq_Xva = seq_dir/"payload_seq_train.npz", seq_dir/"payload_seq_val.npz"
    if not (seq_Xtr.exists() and seq_Xva.exists()):
        raise FileNotFoundError("Payload sequence artefacts missing. Run 01–02 Section 2.1 to build them.")
    X_train, X_val = sp.load_npz(seq_Xtr), sp.load_npz(seq_Xva)
    y_train, y_val = _hydrate_labels(stage_root/"split_preproc",
                                     target_col=globals().get("TARGET_COL","label"))
    return X_train, X_val, y_train, y_val

try:
    Xtr_tfidf, Xva_tfidf, y_train, y_val = _load_seq()
    CNN_AVAILABLE = True; CNN_SKIPPED_REASON = ""
    print("[ok] CNN inputs loaded (sparse)")
except FileNotFoundError as e:
    warnings.warn(str(e))
    CNN_AVAILABLE = False; CNN_SKIPPED_REASON = str(e)
    Xtr_tfidf = Xva_tfidf = y_train = y_val = None

print("[ready] CNN inputs available" if CNN_AVAILABLE else f"[skip] CNN unavailable — {CNN_SKIPPED_REASON}")
print(">>> Section 7.1 — CNN Input Loader: complete")

>>> Section 7.1 — CNN Input Loader: start


[ok] CNN inputs loaded (sparse)
[ready] CNN inputs available
>>> Section 7.1 — CNN Input Loader: complete


## Section 7.2 — CNN Training

Trains a minimal 1D CNN on the TF‑IDF sequences (dense batches). Skips if inputs unavailable.

In [2]:
# ======================================================
# Section 7.2 — CNN Training
# ======================================================
print(">>> Section 7.2 — CNN Training: start")

if not globals().get("CNN_AVAILABLE", False):
    print(f"[skip] CNN unavailable — {globals().get('CNN_SKIPPED_REASON','')}")
else:
    import numpy as np, scipy.sparse as sp, torch, torch.nn as nn, torch.utils.data as tud

    class SparseDataset(tud.Dataset):
        def __init__(self,X,y): self.X=X; self.y=y
        def __len__(self): return self.X.shape[0]
        def __getitem__(self,idx):
            row=self.X[idx]
            if sp.issparse(row): row=row.toarray().ravel()
            return torch.from_numpy(row.astype(np.float32)), torch.tensor(self.y[idx],dtype=torch.long)

    train_ds, val_ds = SparseDataset(Xtr_tfidf,y_train), SparseDataset(Xva_tfidf,y_val)
    train_loader = tud.DataLoader(train_ds,batch_size=64,shuffle=True)
    val_loader   = tud.DataLoader(val_ds,batch_size=256,shuffle=False)

    class TinyCNN(nn.Module):
        def __init__(self,L):
            super().__init__()
            self.conv=nn.Conv1d(1,16,5,padding=2); self.relu=nn.ReLU()
            self.pool=nn.AdaptiveMaxPool1d(64); self.fc=nn.Linear(64*16,2)
        def forward(self,x):
            x=x.unsqueeze(1); x=self.conv(x); x=self.relu(x); x=self.pool(x)
            return self.fc(x.flatten(1))

    input_len = Xtr_tfidf.shape[1]
    model=TinyCNN(input_len); device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device); opt=torch.optim.Adam(model.parameters(),lr=1e-3); crit=nn.CrossEntropyLoss()

    def _epoch(loader,train=True):
        model.train(train); total=correct=0; loss_sum=0.0
        for xb,yb in loader:
            xb,yb=xb.to(device), yb.to(device)
            if train: opt.zero_grad()
            logits=model(xb); loss=crit(logits,yb)
            if train: loss.backward(); opt.step()
            loss_sum+=float(loss.item())*xb.size(0)
            pred=logits.argmax(1); total+=xb.size(0); correct+=int((pred==yb).sum().item())
        return loss_sum/total, correct/total

    best_val=0.0
    for ep in range(3):
        tr_loss,tr_acc=_epoch(train_loader,True)
        va_loss,va_acc=_epoch(val_loader,False)
        best_val=max(best_val,va_acc)
        print(f"[7.2] epoch {ep+1} | train {tr_acc:.3f} | val {va_acc:.3f}")
    globals()["CNN_VAL_ACC"]=best_val
    print(f"[7.2] best validation accuracy: {best_val:.3f}")

print(">>> Section 7.2 — CNN Training: complete")

>>> Section 7.2 — CNN Training: start


[7.2] epoch 1 | train 0.942 | val 0.946


[7.2] epoch 2 | train 0.946 | val 0.946


[7.2] epoch 3 | train 0.946 | val 0.946
[7.2] best validation accuracy: 0.946
>>> Section 7.2 — CNN Training: complete


In [3]:
from pathlib import Path, PurePosixPath
import json
metrics_dir = Path("out") / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)
if "CNN_VAL_ACC" in globals():
    (metrics_dir / "cnn.json").write_text(json.dumps({"val_acc": float(CNN_VAL_ACC)}, indent=2))
    print(f"[persist] wrote CNN metric → {PurePosixPath(metrics_dir/'cnn.json')}")


[persist] wrote CNN metric → out/metrics/cnn.json


In [4]:
# --- 7.3a: Load best tree metric (if available) ---
from pathlib import Path
import json

metrics_path = Path("out") / "metrics" / "best_tree.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        meta = json.load(f)
    BEST_TREE_VAL_ACC = float(meta.get("best_tree_val_acc", 0.0))
    BEST_TREE_NAME    = meta.get("best_tree_name", "tree")
    print(f"[7.3a] loaded best tree: {BEST_TREE_NAME} (val acc={BEST_TREE_VAL_ACC:.3f})")
else:
    print("[7.3a] best_tree.json not found — run 04 and/or 06 with the persist cell added.")

# Now your existing 7.3 block will pick up BEST_TREE_VAL_ACC and show the verdict.


[7.3a] loaded best tree: unknown (val acc=0.000)


## Section 7.3 — CNN vs Tree Baselines

Compares CNN validation accuracy to best tree baseline (if available).

In [5]:
# ======================================================
# Section 7.3 — CNN vs Tree Baselines
# ======================================================
print(">>> Section 7.3 — CNN vs Tree Baselines: start")

cnn_acc=globals().get("CNN_VAL_ACC",None)
tree_acc=globals().get("BEST_TREE_VAL_ACC",None)

def _fmt(x): return "n/a" if x is None else f"{x:.3f}"
print(f"[7.3] CNN val acc:  {_fmt(cnn_acc)}")
print(f"[7.3] Tree val acc: {_fmt(tree_acc)}")

if (cnn_acc is not None) and (tree_acc is not None):
    delta=cnn_acc-tree_acc
    verdict="CNN wins" if delta>0 else "Tree wins" if delta<0 else "Tie"
    print(f"[7.3] verdict: {verdict} (Δ={delta:+.3f})")
else:
    print("[7.3] verdict skipped: missing metrics.")

print(">>> Section 7.3 — CNN vs Tree Baselines: complete")

>>> Section 7.3 — CNN vs Tree Baselines: start
[7.3] CNN val acc:  0.946
[7.3] Tree val acc: 0.000
[7.3] verdict: CNN wins (Δ=+0.946)
>>> Section 7.3 — CNN vs Tree Baselines: complete
